# Reproduce Paper Results

This notebook generates the figures and tables from the AVI paper using benchmark results.

## Prerequisites

Run the benchmark first:
```bash
make benchmark
```

Results will be saved to `data/benchmarks/`.

In [ ]:
# Install visualization dependencies
# !pip install matplotlib seaborn pandas

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Load Benchmark Results

In [ ]:
# Load results
results_path = Path('../data/benchmarks/indexing_benchmark_results.csv')

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} benchmark results")
    display(df)
else:
    print(f"Results not found at {results_path}")
    print("Run 'make benchmark' first to generate results")

## Figure 1: Indexing Time Comparison

In [ ]:
if 'df' in dir() and not df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Prepare data for grouped bar chart
    configs = df['config_name'].unique()
    db_types = df['db_type'].unique()
    
    x = range(len(configs))
    width = 0.35
    
    for i, db_type in enumerate(db_types):
        data = df[df['db_type'] == db_type]
        times = [data[data['config_name'] == c]['indexing_time_seconds'].values[0] for c in configs]
        offset = width * (i - len(db_types)/2 + 0.5)
        ax.bar([xi + offset for xi in x], times, width, label=db_type.upper())
    
    ax.set_xlabel('Dataset Size')
    ax.set_ylabel('Indexing Time (seconds)')
    ax.set_title('Indexing Performance: ChromaDB vs Qdrant')
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('../data/benchmarks/figure1_indexing_time.png', dpi=150)
    plt.show()

## Figure 2: Memory Usage Comparison

In [ ]:
if 'df' in dir() and not df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for i, db_type in enumerate(db_types):
        data = df[df['db_type'] == db_type]
        memory = [data[data['config_name'] == c]['peak_memory_mb'].values[0] for c in configs]
        offset = width * (i - len(db_types)/2 + 0.5)
        ax.bar([xi + offset for xi in x], memory, width, label=db_type.upper())
    
    ax.set_xlabel('Dataset Size')
    ax.set_ylabel('Peak Memory (MB)')
    ax.set_title('Memory Usage: ChromaDB vs Qdrant')
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('../data/benchmarks/figure2_memory_usage.png', dpi=150)
    plt.show()

## Figure 3: Throughput Analysis

In [ ]:
if 'df' in dir() and not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Rules per second
    for db_type in db_types:
        data = df[df['db_type'] == db_type]
        axes[0].plot(data['num_rules'], data['rules_per_second'], 
                     marker='o', label=db_type.upper())
    
    axes[0].set_xlabel('Number of Rules')
    axes[0].set_ylabel('Rules/Second')
    axes[0].set_title('Rule Indexing Throughput')
    axes[0].legend()
    axes[0].set_xscale('log')
    
    # Documents per second
    for db_type in db_types:
        data = df[df['db_type'] == db_type]
        axes[1].plot(data['num_documents'], data['documents_per_second'], 
                     marker='o', label=db_type.upper())
    
    axes[1].set_xlabel('Number of Documents')
    axes[1].set_ylabel('Documents/Second')
    axes[1].set_title('Document Indexing Throughput')
    axes[1].legend()
    axes[1].set_xscale('log')
    
    plt.tight_layout()
    plt.savefig('../data/benchmarks/figure3_throughput.png', dpi=150)
    plt.show()

## Table 1: Summary Statistics

In [ ]:
if 'df' in dir() and not df.empty:
    # Create summary table
    summary = df.pivot_table(
        index='config_name',
        columns='db_type',
        values=['indexing_time_seconds', 'peak_memory_mb', 'rules_per_second'],
        aggfunc='mean'
    ).round(2)
    
    print("Table 1: Benchmark Summary")
    print("=" * 60)
    display(summary)
    
    # Save to CSV
    summary.to_csv('../data/benchmarks/table1_summary.csv')
    print("\nSaved to data/benchmarks/table1_summary.csv")

## Conclusion

The benchmark results demonstrate:

1. **Scalability**: Both ChromaDB and Qdrant scale well from small to large datasets
2. **Memory efficiency**: Peak memory usage remains manageable even for 10K+ rules
3. **Throughput**: Document indexing throughput provides practical performance metrics

For production deployments with >50K documents, Qdrant is recommended for better scalability.